# Component: West Antarctic Ice Sheet (WAIS)

This notebook implements the A4 deep-uncertainty framework for WAIS contributions to GMSL. Key features:
- WAIS is **not** constrained by temperature — scenario-based approach is needed
- A4 framework: three scenarios (S1–S3) with skew-normal distributions in log-space and rheology correction
- S1: Status quo (10%), S2: MISI (80%), S3: MISI+MICI (10%)
- Skewness parameterization follows Robel et al. (2019): MISI amplifies and skews uncertainty
- IMBIE observations show structural break at ~2010
- SSP-independent (WAIS dynamics not directly temperature-driven on projection timescales)

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-poster')

sys.path.insert(0, '.')
from slr_data_readers import read_imbie_west_antarctica
from component_analysis import annualize_imbie
from component_projections import (
    sample_a4_wais, A4_SCENARIOS, RHEOLOGY_FACTOR_MEDIAN,
    RHEOLOGY_FACTOR_SIGMA, _sample_log_skewnormal,
    read_ipcc_component_nc, ipcc_extract,
)
from component_plotting import (
    SSP_COLORS, COMP_COLORS, M_TO_MM,
    plot_component_projection_twopanel,
    plot_component_histogram,
    plot_component_ridge,
)

H5_PATH = '../data/processed/slr_processed_data.h5'
RAW_DIR = '../data/raw'
FIG_DIR = '../figures'
CONF_BASE = f'{RAW_DIR}/ipcc_ar6/slr/ar6/global/confidence_output_files'

BASELINE_YEAR = 2005.0
N_SAMPLES = 2000

## 1. Data Loading

In [ ]:
df_wais = read_imbie_west_antarctica(f'{RAW_DIR}/ice_sheets/imbie_west_antarctica_2021_mm.csv')
wais_year, wais_rebase, wais_sigma = annualize_imbie(df_wais, baseline_year=BASELINE_YEAR)

print(f'WAIS IMBIE: {wais_year[0]:.0f}–{wais_year[-1]:.0f}, {len(wais_year)} points')
print(f'WAIS cumulative at end: {wais_rebase[-1]*M_TO_MM:.1f} mm')

## 2. A4 Framework

### Scenario structure

| Scenario | P | 2100 range (mm) | α | Physics |
|----------|---|-----------------|---|---------|
| S1: Status quo | 10% | 30–80 | 0 | Current discharge rates continue; no MISI |
| S2: MISI | 80% | 150–1000 | +4 | Marine ice sheet instability with amplification |
| S3: MISI+MICI | 10% | 600–2000 | −3 | Full instability cascade including ice cliff failure |

### Distribution
- Skew-normal in log-space: 5th/95th percentiles anchored to low/high bounds; α controls skewness
- S2 positive skew (Robel et al. 2019): MISI amplifies worst-case outcomes
- S3 negative skew: skepticism that MICI operates at maximum efficiency (Morlighem et al. 2024)

### Corrections applied
- **A1 Rheology**: n=3→n=4 correction (Martin et al. 2026), factor 1.28 ± 0.07, applied to all scenarios

In [ ]:
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Per-scenario PDFs at 2100
ax = axes[0]
x_grid = np.linspace(0, 3000, 600)
scenario_colors = {'S1_status_quo': 'C0', 'S2_misi': 'C1', 'S3_misi_mici': 'C3'}

for sname, params in A4_SCENARIOS.items():
    s_rng = np.random.default_rng(hash(sname) % 2**31)
    # Draw from per-scenario distribution (before rheology)
    base = _sample_log_skewnormal(
        200000, params['low_mm'], params['high_mm'], params['alpha'], s_rng,
    )
    # Apply rheology correction
    rheo = s_rng.normal(RHEOLOGY_FACTOR_MEDIAN, RHEOLOGY_FACTOR_SIGMA, 200000)
    rheo = np.maximum(rheo, 1.0)
    base *= rheo

    kde = gaussian_kde(base, bw_method='scott')
    label = f'{sname.replace("_", " ")} (P={params["P"]:.0%}, α={params["alpha"]:.0f})'
    ax.plot(x_grid, kde(x_grid), lw=2, color=scenario_colors[sname], label=label)
    ax.fill_between(x_grid, kde(x_grid), alpha=0.15, color=scenario_colors[sname])

ax.set_xlabel('WAIS SLR at 2100 (mm)')
ax.set_ylabel('Density')
ax.set_title('A4 Scenario PDFs at 2100')
ax.legend(fontsize=7)
ax.set_xlim(0, 3000)
ax.grid(True, alpha=0.2)

# Panel B: Mixture distribution
ax = axes[1]
mixture = sample_a4_wais(200000, np.random.default_rng(99), year=2100)
kde_mix = gaussian_kde(mixture, bw_method='scott')
ax.plot(x_grid, kde_mix(x_grid), 'k-', lw=2, label='A4 mixture')
ax.fill_between(x_grid, kde_mix(x_grid), alpha=0.3, color='gray')
ax.axvline(np.median(mixture), color='k', ls='--', lw=1, label=f'Median: {np.median(mixture):.0f} mm')
p5, p95 = np.percentile(mixture, [5, 95])
ax.axvline(p5, color='gray', ls=':', lw=1)
ax.axvline(p95, color='gray', ls=':', lw=1, label=f'90% CI: [{p5:.0f}, {p95:.0f}] mm')
ax.set_xlabel('WAIS SLR at 2100 (mm)')
ax.set_ylabel('Density')
ax.set_title('A4 Mixture Distribution')
ax.legend(fontsize=8)
ax.set_xlim(0, 3000)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/component_wais_a4_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'A4 mixture at 2100: median={np.median(mixture):.0f} mm, '
      f'mean={np.mean(mixture):.0f} mm, '
      f'[5th, 95th] = [{p5:.0f}, {p95:.0f}] mm')
skew = float(np.mean(((mixture - np.mean(mixture)) / np.std(mixture))**3))
print(f'Skewness: {skew:.2f}')

## 3. Diagnostics & Validation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: IMBIE time series
ax = axes[0]
ax.errorbar(wais_year, wais_rebase * M_TO_MM, yerr=2*wais_sigma*M_TO_MM,
            fmt='o-', ms=4, color=COMP_COLORS['WAIS'], lw=1.5, label='IMBIE WAIS')
ax.axvline(2010, color='red', ls='--', lw=1.5, alpha=0.7, label='Structural break ~2010')
ax.set_xlabel('Year')
ax.set_ylabel('WAIS SLR (mm, 2005 baseline)')
ax.set_title('WAIS observations (IMBIE)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# Panel B: Rate evolution
ax = axes[1]
from component_analysis import compute_component_rates
rates = compute_component_rates(wais_year, wais_rebase, window=2) * M_TO_MM
valid = np.isfinite(rates)
ax.plot(wais_year[valid], rates[valid], 'o-', ms=4, color=COMP_COLORS['WAIS'], lw=1.5)
ax.axvline(2010, color='red', ls='--', lw=1.5, alpha=0.7, label='~2010 break')
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlabel('Year')
ax.set_ylabel('WAIS rate (mm/yr)')
ax.set_title('WAIS rate evolution')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/component_wais_observations.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Projections

In [ ]:
PROJ_YEARS = np.arange(1950, 2151, dtype=float)
PROJ_SSPS = ['SSP1-2.6', 'SSP2-4.5', 'SSP3-7.0', 'SSP5-8.5']
rng_wais = np.random.default_rng(2026)

# A4 is SSP-independent — same samples for all SSPs
wais_samples_mm = np.zeros((N_SAMPLES, len(PROJ_YEARS)))
for j, yr in enumerate(PROJ_YEARS):
    if yr <= BASELINE_YEAR:
        continue  # zero before baseline
    wais_samples_mm[:, j] = sample_a4_wais(N_SAMPLES, rng_wais, year=yr)

wais_samples_m = wais_samples_mm / M_TO_MM

# Build projection dict (same for all SSPs)
wais_proj = {}
for ssp in PROJ_SSPS:
    wais_proj[ssp] = {
        'samples': wais_samples_m,
        'median': np.median(wais_samples_m, axis=0),
        'p5': np.percentile(wais_samples_m, 5, axis=0),
        'p95': np.percentile(wais_samples_m, 95, axis=0),
        'p17': np.percentile(wais_samples_m, 17, axis=0),
        'p83': np.percentile(wais_samples_m, 83, axis=0),
    }

idx_2100 = np.argmin(np.abs(PROJ_YEARS - 2100))
med = np.median(wais_samples_mm[:, idx_2100])
p5 = np.percentile(wais_samples_mm[:, idx_2100], 5)
p95 = np.percentile(wais_samples_mm[:, idx_2100], 95)
print(f'WAIS at 2100 (A4): {med:.0f} [{p5:.0f}, {p95:.0f}] mm (SSP-independent)')

In [ ]:
# WAIS has no temperature dependence — show obs + forward projection
# Upper: WAIS A4 projections with IMBIE obs
# Lower: omit temperature (not applicable)

fig, ax = plt.subplots(figsize=(10, 5))

# Observations
ax.errorbar(wais_year, wais_rebase * M_TO_MM, yerr=2*wais_sigma*M_TO_MM,
            fmt='o', ms=4, color='#444444', lw=1, label='IMBIE observed', zorder=5)

# Projections (SSP-independent, show as single fan)
proj_mask = PROJ_YEARS >= BASELINE_YEAR
yr_plot = PROJ_YEARS[proj_mask]
p = wais_proj['SSP2-4.5']
ax.plot(yr_plot, p['median'][proj_mask] * M_TO_MM, color=COMP_COLORS['WAIS'], lw=2,
        label='A4 projection (median)')
ax.fill_between(yr_plot, p['p17'][proj_mask]*M_TO_MM, p['p83'][proj_mask]*M_TO_MM,
                color=COMP_COLORS['WAIS'], alpha=0.25, label='66% CI')
ax.fill_between(yr_plot, p['p5'][proj_mask]*M_TO_MM, p['p95'][proj_mask]*M_TO_MM,
                color=COMP_COLORS['WAIS'], alpha=0.10, label='90% CI')

ax.set_xlabel('Year')
ax.set_ylabel('WAIS SLR (mm)')
ax.set_title('WAIS — A4 Projections (SSP-independent)')
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(1990, 2150)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/component_wais_projection.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
rng_hist = np.random.default_rng(99)
our_samples = wais_samples_mm[:, idx_2100]

SSP_CODE = {'SSP2-4.5': 'ssp245'}
ipcc_data = read_ipcc_component_nc(CONF_BASE, 'medium_confidence', 'ssp245', 'AIS')
sample_sets = [our_samples]
labels = ['A4 framework (WAIS)']
colors = [COMP_COLORS['WAIS']]
if ipcc_data is not None:
    ex = ipcc_extract(ipcc_data)
    yr_idx = np.argmin(np.abs(ex['years'] - 2100))
    ipcc_med = ex['q50'][yr_idx]
    ipcc_sig = (ex['q95'][yr_idx] - ex['q05'][yr_idx]) / (2 * 1.645)
    ipcc_samples = rng_hist.normal(ipcc_med, ipcc_sig, 10000)
    sample_sets.append(ipcc_samples)
    labels.append('IPCC AR6 (total AIS)')
    colors.append('tab:red')

plot_component_histogram(sample_sets, labels, colors, 'WAIS', year=2100,
                          save_path=f'{FIG_DIR}/component_wais_histogram_2100.png')

In [ ]:
RIDGE_YEARS = list(range(2030, 2110, 10))
rng_ridge = np.random.default_rng(202)

samples_by_year = {}
for yr in RIDGE_YEARS:
    idx_yr = np.argmin(np.abs(PROJ_YEARS - yr))
    samples_by_year[yr] = {'A4 framework': wais_samples_mm[:, idx_yr]}

plot_component_ridge(samples_by_year, 'WAIS', 'SSP-independent',
                      source_colors={'A4 framework': COMP_COLORS['WAIS']},
                      save_path=f'{FIG_DIR}/component_wais_ridge.png')

## 5. IPCC Comparison

The IPCC AR6 AIS component combines EAIS + Peninsula + WAIS. Our A4 framework
produces wider uncertainty ranges than the IPCC medium-confidence AIS, reflecting
the deep uncertainty in marine ice sheet instability that process models
(ISMIP6) may underestimate.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
proj_mask = PROJ_YEARS >= 2005
yr_plot = PROJ_YEARS[proj_mask]

# Our WAIS
p = wais_proj['SSP2-4.5']
ax.plot(yr_plot, p['median'][proj_mask]*M_TO_MM, color=COMP_COLORS['WAIS'], lw=2,
        label='WAIS (A4, this study)')
ax.fill_between(yr_plot, p['p5'][proj_mask]*M_TO_MM, p['p95'][proj_mask]*M_TO_MM,
                color=COMP_COLORS['WAIS'], alpha=0.15)

# IPCC total AIS
for ssp, ssp_code in [('SSP2-4.5', 'ssp245'), ('SSP5-8.5', 'ssp585')]:
    ipcc = read_ipcc_component_nc(CONF_BASE, 'medium_confidence', ssp_code, 'AIS')
    if ipcc is not None:
        ex = ipcc_extract(ipcc)
        color = SSP_COLORS.get(ssp, 'gray')
        ax.plot(ex['years'], ex['q50'], color=color, ls='--', lw=1.5,
                label=f'IPCC total AIS ({ssp})')
        ax.fill_between(ex['years'], ex['q05'], ex['q95'], color=color, alpha=0.05)

ax.set_xlabel('Year')
ax.set_ylabel('SLR contribution (mm)')
ax.set_title('WAIS A4 vs IPCC Total AIS')
ax.legend(fontsize=8)
ax.set_xlim(2005, 2150)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/component_wais_ipcc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Appendix: Sensitivity & Reviewer Q&A

### Scenario weight sensitivity
### Range sensitivity
### Rheology exponent sensitivity

In [ ]:
# ── Tornado diagram: perturb each scenario weight by ±0.05 ──
perturbation = 0.05

fig, ax = plt.subplots(figsize=(10, 6))

base_med = np.median(sample_a4_wais(100000, np.random.default_rng(1), year=2100))

scenarios = list(A4_SCENARIOS.keys())
y_pos = np.arange(len(scenarios))
med_lo, med_hi = [], []

for i, sname in enumerate(scenarios):
    orig_p = A4_SCENARIOS[sname]['P']
    orig_remaining = 1.0 - orig_p

    for direction, storage in [('hi', med_hi), ('lo', med_lo)]:
        if direction == 'hi':
            new_p = min(orig_p + perturbation, 0.95)
            seed = 42 + i
        else:
            new_p = max(orig_p - perturbation, 0.01)
            seed = 142 + i

        remaining = 1.0 - new_p
        probs_mod = []
        for s2 in scenarios:
            if s2 == sname:
                probs_mod.append(new_p)
            else:
                probs_mod.append(A4_SCENARIOS[s2]['P'] * remaining / orig_remaining)

        rng_mod = np.random.default_rng(seed)
        samples_mod = np.zeros(100000)
        scenario_idx = rng_mod.choice(len(scenarios), size=100000, p=probs_mod)
        for k, s2 in enumerate(scenarios):
            mask = scenario_idx == k
            n_s = mask.sum()
            if n_s == 0:
                continue
            s = A4_SCENARIOS[s2]
            base_s = _sample_log_skewnormal(
                n_s, s['low_mm'], s['high_mm'], s['alpha'], rng_mod,
            )
            rheo = rng_mod.normal(RHEOLOGY_FACTOR_MEDIAN, RHEOLOGY_FACTOR_SIGMA, n_s)
            rheo = np.maximum(rheo, 1.0)
            base_s *= rheo
            samples_mod[mask] = base_s
        storage.append(np.median(samples_mod))

# Tornado
for i, sname in enumerate(scenarios):
    ax.barh(y_pos[i], med_hi[i] - base_med, left=base_med, height=0.4,
            color='C3', alpha=0.7)
    ax.barh(y_pos[i], med_lo[i] - base_med, left=base_med, height=0.4,
            color='C0', alpha=0.7)

ax.axvline(base_med, color='k', ls='--', lw=1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels([s.replace('_', ' ') for s in scenarios])
ax.set_xlabel('Median WAIS SLR at 2100 (mm)')
ax.set_title(f'Scenario weight sensitivity (±{perturbation:.0%})')
ax.grid(True, alpha=0.2, axis='x')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/component_wais_tornado.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Range sensitivity: ±20% on (low, high) bounds ──
scenarios = list(A4_SCENARIOS.keys())
probs = [A4_SCENARIOS[s]['P'] for s in scenarios]

print('Range sensitivity (±20% on bounds):')
for factor_label, range_factor in [('-20%', 0.8), ('Base', 1.0), ('+20%', 1.2)]:
    rng_rs = np.random.default_rng(77)
    samples_rs = np.zeros(100000)
    scenario_idx = rng_rs.choice(len(scenarios), size=100000, p=probs)
    for k, sname in enumerate(scenarios):
        mask = scenario_idx == k
        n_s = mask.sum()
        if n_s == 0:
            continue
        s = A4_SCENARIOS[sname]
        base_s = _sample_log_skewnormal(
            n_s, s['low_mm'] * range_factor, s['high_mm'] * range_factor,
            s['alpha'], rng_rs,
        )
        rheo = rng_rs.normal(RHEOLOGY_FACTOR_MEDIAN, RHEOLOGY_FACTOR_SIGMA, n_s)
        rheo = np.maximum(rheo, 1.0)
        base_s *= rheo
        samples_rs[mask] = base_s
    p5, med, p95 = np.percentile(samples_rs, [5, 50, 95])
    print(f'  {factor_label}: median={med:.0f} [{p5:.0f}, {p95:.0f}] mm')

# ── Rheology exponent sensitivity ──
print('\nRheology exponent sensitivity:')
for n_exp, rheo_med in [(3.5, 1.14), (4.0, 1.28), (4.5, 1.44)]:
    rng_rh = np.random.default_rng(88)
    samples_rh = np.zeros(100000)
    scenario_idx = rng_rh.choice(len(scenarios), size=100000, p=probs)
    for k, sname in enumerate(scenarios):
        mask = scenario_idx == k
        n_s = mask.sum()
        if n_s == 0:
            continue
        s = A4_SCENARIOS[sname]
        base_s = _sample_log_skewnormal(
            n_s, s['low_mm'], s['high_mm'], s['alpha'], rng_rh,
        )
        rheo = rng_rh.normal(rheo_med, RHEOLOGY_FACTOR_SIGMA, n_s)
        rheo = np.maximum(rheo, 1.0)
        base_s *= rheo
        samples_rh[mask] = base_s
    p5, med, p95 = np.percentile(samples_rh, [5, 50, 95])
    print(f'  n={n_exp}: rheology factor={rheo_med:.2f}, '
          f'median={med:.0f} [{p5:.0f}, {p95:.0f}] mm')

# ── Skewness sensitivity ──
print('\nSkewness (alpha) sensitivity for S2:')
for alpha_test in [0, 2, 4, 6]:
    rng_sk = np.random.default_rng(55)
    samples_sk = np.zeros(100000)
    scenario_idx = rng_sk.choice(len(scenarios), size=100000, p=probs)
    for k, sname in enumerate(scenarios):
        mask = scenario_idx == k
        n_s = mask.sum()
        if n_s == 0:
            continue
        s = A4_SCENARIOS[sname]
        alpha_use = alpha_test if sname == 'S2_misi' else s['alpha']
        base_s = _sample_log_skewnormal(
            n_s, s['low_mm'], s['high_mm'], alpha_use, rng_sk,
        )
        rheo = rng_sk.normal(RHEOLOGY_FACTOR_MEDIAN, RHEOLOGY_FACTOR_SIGMA, n_s)
        rheo = np.maximum(rheo, 1.0)
        base_s *= rheo
        samples_sk[mask] = base_s
    p5, med, p95 = np.percentile(samples_sk, [5, 50, 95])
    print(f'  alpha={alpha_test}: median={med:.0f} [{p5:.0f}, {p95:.0f}] mm')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# §6  Export results to HDF5
# ══════════════════════════════════════════════════════════════════
from component_io import save_wais

save_wais(
    a4_scenarios=A4_SCENARIOS,
    obs_years=wais_year,
    obs_H=wais_rebase,
    obs_sigma=wais_sigma,
    wais_proj=wais_proj,
    rheology_factor_median=RHEOLOGY_FACTOR_MEDIAN,
    rheology_factor_sigma=RHEOLOGY_FACTOR_SIGMA,
)